Phase 2 : Data Cleaning — CAR-T Clinical Time-Series Dataset

**Xử lí các vấn đề của dữ liệu và chuyển về dạng (patient, day, variable, value)**


**Load data**

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

SRC = r'D:\Downloads\NghienCuuKhoaHoc\CAR-T Therapy\result phase2\original dataset.xlsx'

sheet_names = pd.ExcelFile(SRC).sheet_names
print('Sheets found:', sheet_names)


Sheets found: ['Patient info', 'Hiệu quảSurvival', 'Clinical Raw data', 'CART and B cell follow up']


**Xây dựng hàm hỗ trợ xử lí các vấn đề của dữ liệu**
1. Dữ liệu dạng như < 1.5 thành số + bộ lọc (lấy lớn hơn hoặc lấy bé hơn).
2. Dữ liệu dạng text xuất hiện lồng trong các ô dữ liệu numeric (như MRD/Blast có 'gen bcr âm tính' or 'bệnh tái phát lồng bên trong hoặc line có after HTSC bị lồng bên trong).
3. Chuẩn hoá các dạng dữ liệu Binary như Có/Không thành True/False và các dạng ghi tắt như refac, relap thành Refractory, Relapse.
4. Chuyển các đoạn text n ngày thành dạng numeric n (days).

In [2]:
def parse_threshold(val):
    # (< 1.5) => (1.5 , below_limit) 
    if pd.isna(val):
        return np.nan, np.nan
    if isinstance(val, (int, float)):
        return float(val), 'none'
    s = str(val).strip()
    m = re.match(r'^([<>])\s*([\d.]+)$', s)
    if m:
        sign, num = m.groups()
        return float(num), ('below_limit' if sign == '<' else 'above_limit')
    try:
        return float(s), 'none'
    except ValueError:
        return np.nan, 'non_numeric'


def parse_qualitative_mrd(val):
    if pd.isna(val):
        return np.nan, np.nan
    if isinstance(val, (int, float)):
        return float(val), np.nan
    s = str(val).strip()
    low = s.lower()
    if 'âm tính' in low:           
        return 0.0, f'reported as qualitative negative ("{s}")'
    if 'tái phát' in low:           
        return np.nan, f'not evaluable: {s}'
    try:
        return float(s), np.nan
    except ValueError:
        return np.nan, f'unparsed text: {s}'


def parse_yesno(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in ('có', 'co', 'yes', 'y'):
        return True
    if s in ('không', 'khong', 'no', 'n'):
        return False
    return np.nan


def parse_ngay_duration(val):
    # 6 ngày => 6.0 (days)
    if pd.isna(val):
        return np.nan
    m = re.search(r'(\d+)', str(val))
    return float(m.group(1)) if m else np.nan


def parse_relap_refact(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower().replace(' ', '')
    has_relap, has_refac = 'relap' in s, 'refac' in s
    if has_relap and has_refac:
        return 'relapsed/refractory'
    if has_relap:
        return 'relapsed'
    if has_refac:
        return 'refractory'
    return s


def parse_line(val):
    # 5 (after HSTC) => (5.0, 'after HSTC')
    if pd.isna(val):
        return np.nan, np.nan
    if isinstance(val, (int, float)):
        return float(val), np.nan
    s = str(val).strip()
    m = re.match(r'^(\d+)\s*\(?(.*)\)?$', s)
    if m:
        num, note = m.groups()
        return float(num), (note.strip(') ').strip() or np.nan)
    try:
        return float(s), np.nan
    except ValueError:
        return np.nan, s


DAY_MAP = {
    'Prior apheresis': -13.0,   #số liệu này được lấy từ biểu đồ thời gian trong bài ppt của nhóm Cô Vân     
    'Before CAR T infusion': -1.0,   
    'Day 3/4': 3.5,
    'Day 7+/-1': 7.0,
    'Day 10+/-1': 10.0,
    'Day 14+/-2': 14.0,
    'Day 21+/-2': 21.0,
    'Day 30+/-2': 30.0,
    'Day 60+/-2': 60.0,
    'Day 90+/-14': 90.0,
    'Day 180+/-14': 180.0,
    'Last visit': np.nan,            
}
TIMEPOINT_ORDER = {k: i for i, k in enumerate(DAY_MAP.keys())}
print('Helper functions and Day-grid ready.')


Helper functions and Day-grid ready.


**Làm sạch sheet Clinical Raw Data**
1. Drop các cột không quan trọng hoặc không có tiêu đề, Xoá các cột ngày tháng bị trùng lặp và đổi tên toàn bộ cột sang Tiếng Anh
2. Gọi các helper function lọc qua toàn bộ sheet
3. Xử lí chuỗi thời gian cho đặc trưng IL6, cụ thể là căn cứ vào "ngày phát hiện CRS", "thời gian diễn ra CRS" và "thời điểm dùng Toci sau CRS (h)" để xác định xem các thời điểm IL6 được đo là nằm ở day thứ bao nhiêu kể từ ngày infusion.  


In [3]:
raw1 = pd.read_excel(SRC, sheet_name='Clinical Raw data')
s1 = raw1[raw1['PID'].notna()].copy()

s1 = s1.rename(columns={
    'PID': 'PID', 'Mã BN': 'Patient_code', 'Họ tên': 'Initials',
    'Thời gian nhập viện.1': 'Admission_date',
    'Thời gian truyền tế bào CAR-T.1': 'Infusion_date',
    'Thời gian ra viện.1': 'Discharge_date',
    'Thời gian ký ICF': 'ICF_signed_date',
    'Thời gian nằm viện (tổng)': 'LOS_total_days',
    'Thời gian nằm viện sau điều trị với tế bào CAR-T': 'LOS_post_infusion_days',
    'Chẩn đoán': 'Diagnosis',
    'chẩn đoán trước điều trị': 'Disease_status_pre_raw',
    'PET/CT trước điều trị': 'PETCT_pre', 'PET/CT sau điều trị': 'PETCT_post_tx',
    'Tuổi': 'Age', 'CRS grade': 'CRS_grade',
    'Unnamed: 29': 'CRS_onset_day',           
    'Nhiệt độ': 'Max_temp_C', 'ICANS grade': 'ICANS_grade',
    'Tocilizumab': 'Tocilizumab_given_raw', 'Tocilizumab dose': 'Tocilizumab_dose',
    'Thời điểm dùng Toci sau CRS (h)': 'Time_to_Toci_after_CRS_h',
    'Unnamed: 41': 'Duration_unlabeled_h',     #uname là các cột không có tiêu đề 
    'Steroid': 'Steroid_given_raw',
    'Đáp ứng 1 tháng': 'Response_1m_raw', 'Đáp ứng hiện tại': 'Response_current_raw',
    'Thời điểm bệnh tái phát': 'Relapse_date_raw', 'Thời điểm tử vong': 'Death_date_raw',
})


s1 = s1.drop(columns=[
    'STT', 'Thời gian nhập viện', 'Thời gian truyền tế bào CAR-T', 'Thời gian ra viện',
    'Unnamed: 42',
    'Thời gian kết thúc điều trị bệnh nhân đáp ứng hoàn toàn',
    'Sống thêm tổng thể (OS) sau khi truyền tế bào CD19 CAR-T',
])

for col in ['MRD trước điều trị', 'MRD 1 tháng', 'MRD 3 tháng', 'MRD 6 tháng',
            'Blast tủy trước điều trị', 'Blast tủy 1 tháng', 'Blast tủy 3 tháng', 'Blast tủy 6 tháng']:
    parsed = s1[col].apply(parse_qualitative_mrd)
    newname = (col.replace('MRD trước điều trị', 'MRD_pre')
                  .replace('MRD 1 tháng', 'MRD_1m').replace('MRD 3 tháng', 'MRD_3m')
                  .replace('MRD 6 tháng', 'MRD_6m')
                  .replace('Blast tủy trước điều trị', 'Blast_pct_pre')
                  .replace('Blast tủy 1 tháng', 'Blast_pct_1m').replace('Blast tủy 3 tháng', 'Blast_pct_3m')
                  .replace('Blast tủy 6 tháng', 'Blast_pct_6m'))
    s1[newname] = parsed.apply(lambda t: t[0])
    s1[newname + '_note'] = parsed.apply(lambda t: t[1])
    s1 = s1.drop(columns=[col])

for col, newname in [('IL6 trước truyền CAR-T', 'IL6_pre_infusion'), ('IL6 khi bắt đầu CRS', 'IL6_CRS_onset'),
                      ('IL6 trước khi dùng Toci', 'IL6_pre_Toci'), ('IL6 cao nhất', 'IL6_peak'),
                      ('IL6 khi kết thúc CRS', 'IL6_CRS_end')]:
    parsed = s1[col].apply(parse_threshold)
    s1[newname] = parsed.apply(lambda t: t[0])
    s1[newname + '_censor'] = parsed.apply(lambda t: t[1])
    s1 = s1.drop(columns=[col])

s1['Tocilizumab_given'] = s1['Tocilizumab_given_raw'].apply(parse_yesno)
s1['Steroid_given'] = s1['Steroid_given_raw'].apply(parse_yesno)
s1 = s1.drop(columns=['Tocilizumab_given_raw', 'Steroid_given_raw'])

s1['CRS_duration_days'] = s1['Thời gian diễn ra CRS'].apply(parse_ngay_duration)
s1 = s1.drop(columns=['Thời gian diễn ra CRS', 'Ngày phát hiện CRS'])

s1['Disease_status_pre'] = s1['Disease_status_pre_raw'].apply(parse_relap_refact)
s1 = s1.drop(columns=['Disease_status_pre_raw'])

parsed_line = s1['Line'].apply(parse_line)
s1['Line_of_therapy'] = parsed_line.apply(lambda t: t[0])
s1['Line_note'] = parsed_line.apply(lambda t: t[1])
s1 = s1.drop(columns=['Line'])

s1['PID'] = s1['PID'].astype(int)
s1['CRS_grade'] = pd.to_numeric(s1['CRS_grade'], errors='coerce')

# Đảm bảo cột thời gian dùng Toci ở định dạng số
s1['Time_to_Toci_after_CRS_h'] = pd.to_numeric(s1['Time_to_Toci_after_CRS_h'], errors='coerce')

# Tính ngày dùng Toci = Ngày bắt đầu CRS + (Số giờ / 24)
toci_day = s1['CRS_onset_day'] + (s1['Time_to_Toci_after_CRS_h'] / 24)

# Tạo bảng time-series cho riêng IL6_pre_Toci
toci_long = pd.DataFrame({
    'PID': s1['PID'],
    'Day': toci_day,
    'IL6_pre_Toci': s1['IL6_pre_Toci']
}).dropna(subset=['Day', 'IL6_pre_Toci'])

# Xử lý các ca trùng ngày (nếu có) bằng cách lấy giá trị trung bình
toci_long = toci_long.groupby(['PID', 'Day'], as_index=False).mean()

print('Sheet1 cleaned:', s1.shape)
s1[['PID', 'Patient_code', 'Diagnosis', 'Age', 'Disease_status_pre', 'CRS_grade', 'Tocilizumab_given']].head()

# 1. Melt các giá trị IL-6
il6_melted = s1.melt(
    id_vars=['PID', 'CRS_onset_day', 'CRS_duration_days'], 
    value_vars=['IL6_pre_infusion', 'IL6_CRS_onset', 'IL6_CRS_end'],
    var_name='Event_Type', value_name='IL6'
)

# 2. Melt các cờ đánh dấu (censor) tương ứng
censor_melted = s1.melt(
    id_vars=['PID'], 
    value_vars=['IL6_pre_infusion_censor', 'IL6_CRS_onset_censor', 'IL6_CRS_end_censor'],
    value_name='IL6_censor'
)
il6_melted['IL6_censor'] = censor_melted['IL6_censor']

# 3. Tính toán Ngày thực tế cho từng sự kiện
def map_il6_day(row):
    if row['Event_Type'] == 'IL6_pre_infusion': return -1.0
    if row['Event_Type'] == 'IL6_CRS_onset': return row['CRS_onset_day']
    if row['Event_Type'] == 'IL6_CRS_end': return row['CRS_onset_day'] + row['CRS_duration_days']
    return np.nan

il6_long_temp = il6_melted.copy()
il6_long_temp['Day'] = il6_long_temp.apply(map_il6_day, axis=1)
il6_long_temp = il6_long_temp.dropna(subset=['Day', 'IL6'])[['PID', 'Day', 'IL6', 'IL6_censor']]

# 4. Gộp nhóm các trường hợp bị trùng ngày 
# (Vd: Bệnh nhân bắt đầu và kết thúc CRS trong cùng 1 ngày thì lấy giá trị trung bình)
global_il6_long = il6_long_temp.groupby(['PID', 'Day'], as_index=False).agg({'IL6': 'mean', 'IL6_censor': 'first'})


Sheet1 cleaned: (16, 56)


**Chuyển đổi dữ liệu theo dõi car-t và B-cell theo dạng trục thời gian dọc**

In [4]:
raw2 = pd.read_excel(SRC, sheet_name='CART and B cell follow up', header=None)

junk_block = raw2.iloc[5:24, 63:73] #kiểm tra vùng dữ liệu rác, nó chỉ là tiêu đề bị lặp lại => drop
print('Vùng dữ liệu rác', junk_block.notna().sum().sum(),
      '-> drop')

group_defs = [
    ('B_cell_PB_count',  range(4, 16)),
    ('B_cell_PB_pct',    range(16, 28)),
    ('CAR_T_PB_count',   range(28, 38)),
    ('CAR_T_PB_pct',     range(38, 48)),
    ('B_cell_BM_count',  range(48, 52)),
    ('B_cell_BM_pct',    range(52, 56)),
    ('CAR_T_BM_count',   range(56, 59)),
    ('CAR_T_BM_pct',     range(59, 62)),
]
header_row = raw2.iloc[4]
data2 = raw2.iloc[5:24].reset_index(drop=True)

id_cols = data2[[0, 1, 2, 3]].copy()
id_cols.columns = ['Project_ID', 'Patient_initials', 'PID', 'Diagnosis']
id_cols['PID'] = id_cols['PID'].astype(int)

long_frames = []
for var_name, col_range in group_defs:
    for c in col_range:
        timepoint = header_row[c]
        long_frames.append(pd.DataFrame({
            'PID': id_cols['PID'], 'Timepoint_label': timepoint,
            'Variable': var_name, 'Value': pd.to_numeric(data2[c], errors='coerce'),
        }))

s2_long = pd.concat(long_frames, ignore_index=True)
s2_long['Day'] = s2_long['Timepoint_label'].map(DAY_MAP)
s2_long['Timepoint_order'] = s2_long['Timepoint_label'].map(TIMEPOINT_ORDER)

s2_long = s2_long.dropna(subset=['Value']).reset_index(drop=True)

s2_wide = s2_long.pivot_table(index=['PID', 'Timepoint_label', 'Day', 'Timepoint_order'],
                               columns='Variable', values='Value', aggfunc='first').reset_index()
s2_wide = s2_wide.merge(id_cols[['PID', 'Patient_initials', 'Diagnosis']].drop_duplicates(), on='PID', how='left')

s2_wide.sort_values(['PID', 'Day']).head(8)


Vùng dữ liệu rác 1 -> drop


,PID,Timepoint_label,Day,Timepoint_order,B_cell_BM_count,B_cell_BM_pct,B_cell_PB_count,B_cell_PB_pct,CAR_T_BM_count,CAR_T_BM_pct,CAR_T_PB_count,CAR_T_PB_pct,Patient_initials,Diagnosis
10,201059250,Prior apheresis,-13.0,0,NaN,NaN,393000.0,11.69,NaN,NaN,NaN,NaN,NTH,NHL
0,201059250,Before CAR T infusion,-1.0,1,NaN,NaN,7530.0,0.47,NaN,NaN,NaN,NaN,NTH,NHL
5,201059250,Day 3/4,3.5,2,NaN,NaN,1510.0,0.22,NaN,NaN,0.534831,0.2559,NTH,NHL
8,201059250,Day 7+/-1,7.0,3,NaN,NaN,100.0,0.00,NaN,NaN,705.531250,33.4375,NTH,NHL
1,201059250,Day 10+/-1,10.0,4,NaN,NaN,0.0,0.00,NaN,NaN,629.314890,48.7841,NTH,NHL
2,201059250,Day 14+/-2,14.0,5,NaN,NaN,150.0,0.00,NaN,NaN,2277.495450,54.3555,NTH,NHL
4,201059250,Day 21+/-2,21.0,6,NaN,NaN,251.0,0.00,NaN,NaN,1629.598080,46.2954,NTH,NHL
6,201059250,Day 30+/-2,30.0,7,NaN,NaN,301.0,0.01,NaN,NaN,425.499180,22.8763,NTH,NHL


**Làm sạch sheet Hiệu quả survival và xử lí chuỗi thời gian cho đặc trưng Response và IgG**
1. Đổi tên các cột thành tiếng Anh
2. Gắn các kết quả CR, PR, No Response, Relapse cho các thời điểm tương ứng ở lưới thời gian có sẵn gồm D3,D7,....
3. Riêng đối với Die của Response thì lấy dữ liệu thời điểm ở cột "thời điểm tử vong" - "infusion_date" để ra số days => gắn giá trị "Die" vô thời điểm tại Day mà mình tính ra (Nếu thời điểm đó out khỏi cái day_map thì tự tạo một dòng mới để thể hiện)
4. Đối với chỉ số IgG thì cũng trích xuất ra thời điểm đo trong cột "xét nghiệm IgG..." - "infusion_date" để ra gắn vào trục thời gian (Mục này cũng sẽ tạo 1 dòng mới tương tự nếu như gặp trường hợp thời điểm ở ngoài day_map)

In [5]:
raw3 = pd.read_excel(SRC, sheet_name='Hiệu quảSurvival', header=None)
header3 = raw3.iloc[1]
data3 = raw3.iloc[2:].copy()
data3.columns = header3
s3 = data3[data3['Mã BN'].notna()].reset_index(drop=True)

s3 = s3.rename(columns={
    'Mã BN': 'Patient_code', 'Tên BN': 'Initials', 'Ngày truyền': 'Infusion_date_s3',
    'Liều truyền (million cells/kg)': 'Dose_million_cells_per_kg',
    'Đáp ứng D30': 'Response_D30', 'Đáp ứng D90': 'Response_D90', ' Đáp ứng D180': 'Response_D180',
    'Tình trạng hiện tại (May 2025)': 'Status_current',
    'Ngày tái phát (dd/mm/yy)': 'Relapse_date', 'Số ngày tính đến ngày tái phát': 'Days_to_relapse',
    'Sống (hiện tại)': 'Alive_now_raw', 'Sống (6 tháng) ': 'Alive_6m_raw', 'Thời điểm tử vong': 'Death_date',
    'Thời gian theo dõi (ngày) ': 'Followup_days', 'Thời gian theo dõi (tháng) ': 'Followup_months',
    'Baseline IgG (mg/dL)': 'IgG_baseline',
    'Xét nghiệm IgG (mg/ dL) (trên 700 là hồi phục)': 'IgG_series_raw',
    'Ngày hồi phục IgG': 'IgG_recovery_date', 'Hồi phục IgG': 'IgG_recovered_raw',
    'Blast trước điều trị ': 'Blast_pct_pre_s3',
})[['Patient_code', 'Initials', 'Infusion_date_s3', 'Dose_million_cells_per_kg',
    'Response_D30', 'Response_D90', 'Response_D180', 'Status_current',
    'Relapse_date', 'Days_to_relapse', 'Alive_now_raw', 'Alive_6m_raw', 'Death_date',
    'Followup_days', 'Followup_months', 'IgG_baseline', 'IgG_series_raw',
    'IgG_recovery_date', 'IgG_recovered_raw', 'Blast_pct_pre_s3']]

s3['Alive_now'] = s3['Alive_now_raw'].apply(parse_yesno)
s3['Alive_6m'] = s3['Alive_6m_raw'].apply(parse_yesno)
s3['IgG_recovered'] = s3['IgG_recovered_raw'].apply(parse_yesno)
s3 = s3.drop(columns=['Alive_now_raw', 'Alive_6m_raw', 'IgG_recovered_raw'])

code_to_pid = s1[['Patient_code', 'PID']].drop_duplicates()
s3 = s3.merge(code_to_pid, on='Patient_code', how='left')

# 1. Melt các đánh giá định kỳ (CR, PR, No Response, Relapse)
resp_periodic = s3.melt(id_vars=['PID'], value_vars=['Response_D30', 'Response_D90', 'Response_D180'],
                        var_name='Timepoint_label', value_name='Response_raw')
resp_periodic['Day'] = resp_periodic['Timepoint_label'].map({'Response_D30': 30.0, 'Response_D90': 90.0, 'Response_D180': 180.0})
resp_periodic = resp_periodic.dropna(subset=['Response_raw']).reset_index(drop=True)

# Chuẩn hóa nhãn định kỳ
def clean_periodic_label(raw):
    raw_str = str(raw).lower()
    if 'cr' in raw_str or 'hoàn toàn' in raw_str: return 'CR'
    if 'pr' in raw_str or 'một phần' in raw_str: return 'PR'
    if 'no response' in raw_str or 'không đáp ứng' in raw_str: return 'No Response'
    if 'tái phát' in raw_str or 'relapse' in raw_str: return 'Relapse'
    return np.nan

resp_periodic['Response'] = resp_periodic['Response_raw'].apply(clean_periodic_label)
resp_periodic = resp_periodic.dropna(subset=['Response'])[['PID', 'Day', 'Response']]

# Sử dụng dữ liệu ở cột Thời điểm tử vong (Death_date)
# Merge với s1 để lấy Infusion_date sau đó tính toán ra thời điểm tử vong ở day nào
s3_terminal = s3.merge(s1[['PID', 'Infusion_date']], on='PID', how='left')

terminal_rows = []
for _, row in s3_terminal.iterrows():
    # Kiểm tra xem bệnh nhân có giá trị ở cột Thời điểm tử vong hay không
    if pd.notna(row['Death_date']) and pd.notna(row['Infusion_date']):
        try:
            # Chuyển đổi sang định dạng ngày tháng để tính toán
            d_date = pd.to_datetime(row['Death_date'])
            # Lấy thời điểm tử vong trừ đi ngày truyền CAR-T
            exact_day = float((d_date - row['Infusion_date']).days)
            # Tạo một dòng dữ liệu mới với nhãn "Die"
            terminal_rows.append({'PID': row['PID'], 'Day': exact_day, 'Response': 'Die', 'Timepoint_label': 'Terminal_Event'})
        except Exception:
            pass

resp_terminal = pd.DataFrame(terminal_rows)

# 3. Gom chung đánh giá định kỳ và sự kiện Tử vong
if not resp_terminal.empty:
    resp_long = pd.concat([resp_periodic, resp_terminal], ignore_index=True)
else:
    resp_long = resp_periodic.copy()
    resp_long['Timepoint_label'] = np.nan

#Đảm bảo không có dữ liệu phi logic (cụ thể là sau ngày tử vong mà vẫn còn record)
valid_rows = []
for pid, group in resp_long.groupby('PID'):
    group = group.sort_values('Day')
    
    terminal_days = group[group['Response'].isin(['Die'])]['Day']
    if not terminal_days.empty:
        cutoff_day = terminal_days.min()
        # Chỉ giữ lại các bản ghi có Ngày <= ngày Tử vong
        valid_rows.append(group[group['Day'] <= cutoff_day])
    else:
        valid_rows.append(group)

resp_long = pd.concat(valid_rows).drop_duplicates(subset=['PID', 'Day', 'Response']).reset_index(drop=True)

def parse_igg_series(text):
    if pd.isna(text):
        return []
    out = []
    for chunk in str(text).split('\n'):
        chunk = chunk.strip()
        m = re.match(r'(\d{1,2}/\d{1,2}/\d{4}):\s*([\d.]+)', chunk)
        if m:
            date_str, val = m.groups()
            try:
                out.append((pd.to_datetime(date_str, dayfirst=True), float(val)))
            except Exception:
                pass
    return out


igg_rows = []
for _, row in s3.iterrows():
    for d, val in parse_igg_series(row['IgG_series_raw']):
        igg_rows.append({'PID': row['PID'], 'Date': d, 'IgG_mg_dL': val})
igg_long = pd.DataFrame(igg_rows).merge(s1[['PID', 'Infusion_date']], on='PID', how='left')
igg_long['Day'] = (igg_long['Date'] - igg_long['Infusion_date']).dt.days.astype(float)
igg_long = igg_long.drop(columns=['Infusion_date'])

igg_long.head()


,PID,Date,IgG_mg_dL,Day
0,201264034,2023-12-12,450.0,31.0
1,201264034,2023-12-25,361.0,44.0
2,201264034,2024-01-02,675.0,52.0
3,201264034,2024-01-25,380.0,75.0
4,201264034,2024-05-09,120.0,180.0


**Ghép các bảng đã hoàn thiện lại với nhau**

In [6]:
COARSE_DAY_MAP = {'pre': -1.0, '1m': 30.0, '3m': 90.0, '6m': 180.0}

def coarse_long(prefix, valuecol_name):
    frames = [pd.DataFrame({'PID': s1['PID'], 'Day': day, valuecol_name: s1[f'{prefix}_{suffix}']})
              for suffix, day in COARSE_DAY_MAP.items()]
    return pd.concat(frames, ignore_index=True).dropna(subset=[valuecol_name])

mrd_long = coarse_long('MRD', 'MRD')
blast_long = coarse_long('Blast_pct', 'Blast_pct')

# --- HỢP NHẤT TOÀN BỘ CÁC BẢNG ---
master = s2_wide.copy()
master = master.merge(mrd_long, on=['PID', 'Day'], how='outer')
master = master.merge(blast_long, on=['PID', 'Day'], how='outer')
master = master.merge(resp_long[['PID', 'Day', 'Response']], on=['PID', 'Day'], how='outer')
master = master.merge(igg_long[['PID', 'Day', 'IgG_mg_dL']], on=['PID', 'Day'], how='outer')

# Merge IL-6 (Lấy từ biến global_il6_long đã tạo ở Phần 3)
master = master.merge(global_il6_long, on=['PID', 'Day'], how='outer')
# Il6-preToci
master = master.merge(toci_long, on=['PID', 'Day'], how='outer')

inv_day_map = {v: k for k, v in DAY_MAP.items() if pd.notna(v)}
inv_day_map[-1.0] = 'Before CAR T infusion'
master['Timepoint_label'] = master['Timepoint_label'].fillna(master['Day'].map(inv_day_map))
master['Timepoint_order'] = master['Timepoint_order'].fillna(master['Timepoint_label'].map(TIMEPOINT_ORDER))

diag_map = (pd.concat([s1[['PID', 'Diagnosis']], id_cols[['PID', 'Diagnosis']]])
            .drop_duplicates('PID').set_index('PID')['Diagnosis'])
master['Diagnosis'] = master['PID'].map(diag_map)

baseline_cols = ['PID', 'Patient_code', 'Age', 'Line_of_therapy', 'Line_note', 'Disease_status_pre',
                 'CRS_grade', 'CRS_onset_day', 'CRS_duration_days', 'ICANS_grade',
                 'Tocilizumab_given', 'Tocilizumab_dose', 'Steroid_given', 'IL6_peak']
master = master.merge(s1[baseline_cols], on='PID', how='left')
master = master.sort_values(['Patient_code', 'PID', 'Day']).reset_index(drop=True)

# Gán thứ tự cột mới như yêu cầu
col_order = ['PID', 'Patient_code', 'Age', 'Diagnosis', 'Line_of_therapy', 'Line_note', 
             'Disease_status_pre', 'Day', 'Timepoint_label', 'Timepoint_order',
             'CAR_T_PB_count', 'CAR_T_PB_pct', 'CAR_T_BM_count', 'CAR_T_BM_pct',
             'B_cell_PB_count', 'B_cell_PB_pct', 'B_cell_BM_count', 'B_cell_BM_pct',
             'MRD', 'Blast_pct', 'IgG_mg_dL', 'IL6', 'IL6_censor', 'IL6_pre_Toci', 'IL6_peak', 
             'CRS_grade', 'CRS_onset_day', 'CRS_duration_days', 'ICANS_grade',
             'Tocilizumab_given', 'Tocilizumab_dose', 'Steroid_given', 'Response']
master = master[col_order]

print('MASTER table:', master.shape)
master.head(10)

MASTER table: (279, 33)


,PID,Patient_code,Age,Diagnosis,Line_of_therapy,Line_note,Disease_status_pre,Day,Timepoint_label,Timepoint_order,CAR_T_PB_count,CAR_T_PB_pct,CAR_T_BM_count,CAR_T_BM_pct,B_cell_PB_count,B_cell_PB_pct,B_cell_BM_count,B_cell_BM_pct,MRD,Blast_pct,IgG_mg_dL,IL6,IL6_censor,IL6_pre_Toci,IL6_peak,CRS_grade,CRS_onset_day,CRS_duration_days,ICANS_grade,Tocilizumab_given,Tocilizumab_dose,Steroid_given,Response
0,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,-13.0,Prior apheresis,0.0,NaN,NaN,NaN,NaN,5250.0,0.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
1,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,-1.0,Before CAR T infusion,1.0,NaN,NaN,NaN,NaN,3210.0,0.11,NaN,NaN,NaN,0.05,NaN,1.5,below_limit,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
2,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,3.5,Day 3/4,2.0,0.649635,0.2415,NaN,NaN,4260.0,0.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
3,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.0,none,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
4,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,7.0,Day 7+/-1,3.0,143.104964,24.0100,NaN,NaN,1210.0,0.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
5,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,none,65.0,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
6,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,10.0,Day 10+/-1,4.0,528.858200,40.6814,NaN,NaN,150.0,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
7,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,14.0,Day 14+/-2,5.0,409.190000,38.9700,NaN,NaN,301.0,0.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
8,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,21.0,Day 21+/-2,6.0,245.000000,19.0000,NaN,NaN,2210.0,0.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,NaN
9,201217917,BN01,8,ALL,3.0,NaN,relapsed/refractory,30.0,Day 30+/-2,7.0,65.857470,3.0919,74.35191,3.4907,201.0,0.00,6460.0,0.04,NaN,0.00,NaN,NaN,NaN,NaN,109.0,1.0,6.0,2.0,0.0,True,1.0,False,CR


**Thống kê tỉ lệ khuyết và Tạo báo hiệu missing**
- Mỗi biến trong danh sách CAR_T_PB_count', 'CAR_T_PB_pct', 'B_cell_PB_count', 'B_cell_PB_pct','CAR_T_BM_count', 'B_cell_BM_count', 'MRD', 'Blast_pct', 'IgG_mg_dL', 'IL6', 'Response sẽ có 1 cờ báo hiệu mang dạng biến_missing. Ở tại mỗi dòng (tức là mỗi ngày) thì các cờ này sẽ báo hiệu rằng biến đó có bị miss ở ngày hôm nay hay không. Nếu có thì missing sẽ là 1, còn không bị miss thì sẽ là 0. 
Tác dụng : 
- Chuẩn bị đầu vào cho các mô hình nếu không chấp nhận các giá trị rỗng
- Biến missing này em thêm vào là để cho việc mô hình đánh giá một giá trị bị rỗng trở nên sáng suốt hơn. Khi nó nhận về nhiều record về bệnh nhân thiếu giá trị này nhưng vẫn khoẻ, nó sẽ coi việc missing = 1 ở giá trị này là một yếu tố tốt và sẽ gán trọng số cao cho cái cờ này. Còn nếu mà biến này missing là do ngẫu nhiên hay quên đo thì sẽ dẫn tới không có quy luật hay không vẫn dẫn tới một kết quả nào đó thì nó sẽ coi nhẹ biến này đi. Em cảm nhận rằng việc này tốt hơn khi so với việc để trống giá trị ở ô đó và để AI bỏ qua nó khi quét thấy giá trị rỗng

In [7]:
miss_pct = master.isna().mean().sort_values(ascending=False) * 100
print('Missingness by column (%):')
print(miss_pct.round(1).to_string())

key_vars = ['CAR_T_PB_count', 'CAR_T_PB_pct', 'B_cell_PB_count', 'B_cell_PB_pct',
            'CAR_T_BM_count', 'B_cell_BM_count', 'MRD', 'Blast_pct', 'IgG_mg_dL', 'IL6', 'Response']
for v in key_vars:
    master[f'{v}_missing'] = master[v].isna().astype(int)

print('\nRow-level completeness (patients with the most complete records):')
row_completeness = master.set_index(['PID', 'Day'])[key_vars].notna().sum(axis=1)
print(row_completeness.groupby('PID').mean().sort_values(ascending=False).round(2).to_string())


Missingness by column (%):
IL6_pre_Toci          96.8
Line_note             94.3
MRD                   92.1
Blast_pct             90.0
CAR_T_BM_pct          87.5
CAR_T_BM_count        87.5
B_cell_BM_pct         86.0
B_cell_BM_count       86.0
IL6                   84.9
IL6_censor            84.9
Response              84.6
IgG_mg_dL             79.6
Tocilizumab_dose      46.2
CAR_T_PB_pct          41.6
CAR_T_PB_count        41.6
B_cell_PB_pct         28.7
B_cell_PB_count       28.7
Timepoint_order       28.3
Timepoint_label       28.3
CRS_duration_days     24.0
CRS_onset_day         24.0
ICANS_grade            9.0
Age                    9.0
Patient_code           9.0
Line_of_therapy        9.0
Tocilizumab_given      9.0
CRS_grade              9.0
Disease_status_pre     9.0
IL6_peak               9.0
Steroid_given          9.0
PID                    0.0
Diagnosis              0.0
Day                    0.0

Row-level completeness (patients with the most complete records):
PID
201182298  

**Khử nhiễu các số thập phân**

In [8]:
pct_cols = [c for c in master.columns if c.endswith('_pct') or c in ('MRD', 'Blast_pct')]
count_cols = [c for c in master.columns if c.endswith('_count')]
other_float_cols = ['IgG_mg_dL', 'CRS_duration_days', 'Tocilizumab_dose', 'IL6', 'IL6_pre_Toci', 'IL6_peak']

master[pct_cols] = master[pct_cols].round(4)
master[count_cols] = master[count_cols].round(7)
master[other_float_cols] = master[other_float_cols].round(2)

**Xuất file kết quả**

In [9]:
OUT = r'D:\Downloads\NghienCuuKhoaHoc\CAR-T Therapy\result phase2\cart_cleaned_2412357.csv'
master.to_csv(OUT, index=False, encoding='utf-8-sig')
print('Saved:', OUT)
print('Final shape:', master.shape)
master.dtypes


Saved: D:\Downloads\NghienCuuKhoaHoc\CAR-T Therapy\result phase2\cart_cleaned_2412357.csv
Final shape: (279, 44)


PID                          int64
Patient_code                   str
Age                         object
Diagnosis                      str
Line_of_therapy            float64
Line_note                      str
Disease_status_pre             str
Day                        float64
Timepoint_label                str
Timepoint_order            float64
CAR_T_PB_count             float64
CAR_T_PB_pct               float64
CAR_T_BM_count             float64
CAR_T_BM_pct               float64
B_cell_PB_count            float64
B_cell_PB_pct              float64
B_cell_BM_count            float64
B_cell_BM_pct              float64
MRD                        float64
Blast_pct                  float64
IgG_mg_dL                  float64
IL6                        float64
IL6_censor                     str
IL6_pre_Toci               float64
IL6_peak                   float64
CRS_grade                  float64
CRS_onset_day              float64
CRS_duration_days          float64
ICANS_grade         